# 5. Full real-dataset benchmark

This notebook runs the benchmark command used for the KDD tutorial demo.

For a full local run, use all datasets below. For Colab, consider `--max-items 100` or only `longmemeval_oracle.json`.

## Proposal benchmarking design

This notebook implements the standardized benchmarking pipeline:

1. **INGEST:** load experimental data into memory records.
2. **INDEX:** build lexical/offline retrieval indices through `MemoryStore`.
3. **SEARCH:** execute queries and retrieve relevant context.
4. **ANSWER:** produce evidence-available or insufficient-evidence decisions.
5. **EVALUATE:** compare retrieved IDs against ground-truth evidence/context IDs and optionally run semantic evaluators.
6. **REPORT:** aggregate precision, recall, hit rate, utilization, latency, cost, failure modes, and semantic scores.

Provider mapping in this implementation:
- Memory providers: verbatim, extracted facts, episodic, hybrid.
- Evaluators: offline evidence-ID evaluator plus optional offline semantic, Rhesis, and Semantica backends.
- Benchmarks: LoCoMo, LongMemEval, MemoryArena. LCBench/HPOBench are optional extensions.

In [1]:
from pathlib import Path
import json
import os
import sys

# Colab guidance:
# 1. Upload or clone this repository into the Colab runtime.
# 2. Set PROJECT_ROOT to the repository root if auto-detection fails.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "experiment" / "run.py").exists():
    candidates = [Path("/content/kdd"), Path("/content/drive/MyDrive/kdd"), Path("/Users/syaikhipin/Documents/Phd/Publish Paper/kdd")]
    PROJECT_ROOT = next((p for p in candidates if (p / "experiment" / "run.py").exists()), PROJECT_ROOT)

EXPERIMENT_DIR = PROJECT_ROOT / "experiment"
RESULTS_DIR = EXPERIMENT_DIR / "results"
sys.path.insert(0, str(EXPERIMENT_DIR))
print("PROJECT_ROOT =", PROJECT_ROOT)
print("Experiment code exists:", (EXPERIMENT_DIR / "run.py").exists())

PROJECT_ROOT = /Users/syaikhipin/Documents/Phd/Publish Paper/kdd
Experiment code exists: True


In [2]:
import subprocess
cmd = [
    sys.executable, str(EXPERIMENT_DIR / "run.py"),
    "--mode", "real",
    "--backend", "offline",
    "--datasets", "locomo", "longmemeval", "memoryarena",
    "--max-conversations", "999",
    "--max-questions", "999",
    "--top-k", "5",
    "--eval-backend", "offline",
    "--eval-limit", "50",
    "--visualize",
]
print(" ".join(map(str, cmd)))
# To keep notebook reruns fast, only execute if RUN_FULL_BENCHMARK=1.
if os.environ.get("RUN_FULL_BENCHMARK") == "1":
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=600)
    print(result.stdout)
    print(result.stderr)
    assert result.returncode == 0
else:
    print("Skipped full benchmark. Set RUN_FULL_BENCHMARK=1 to execute.")

/Users/syaikhipin/anaconda3/bin/python /Users/syaikhipin/Documents/Phd/Publish Paper/kdd/experiment/run.py --mode real --backend offline --datasets locomo longmemeval memoryarena --max-conversations 999 --max-questions 999 --top-k 5 --eval-backend offline --eval-limit 50 --visualize
Skipped full benchmark. Set RUN_FULL_BENCHMARK=1 to execute.


In [3]:
modal_cmd = [
    sys.executable, str(EXPERIMENT_DIR / "run.py"),
    "--runner", "modal",
    "--modal-gpu", os.environ.get("MODAL_GPU", "T4"),
    "--modal-detach",
    "--mode", "real",
    "--backend", "offline",
    "--datasets", "locomo", "longmemeval", "memoryarena",
    "--max-conversations", "999",
    "--max-questions", "999",
    "--top-k", "5",
    "--eval-backend", "offline",
    "--visualize",
]
print("Paper-quality Modal GPU command:")
print(" ".join(map(str, modal_cmd)))
if os.environ.get("RUN_MODAL_FULL_BENCHMARK") == "1":
    result = subprocess.run(modal_cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=3600)
    print(result.stdout)
    print(result.stderr)
    assert result.returncode == 0
else:
    print("Skipped Modal benchmark. Set RUN_MODAL_FULL_BENCHMARK=1 after Modal setup to execute.")
    print("Detached Modal runs print a call id; fetch with --modal-call-id <call-id>.")

Paper-quality Modal GPU command:
/Users/syaikhipin/anaconda3/bin/python /Users/syaikhipin/Documents/Phd/Publish Paper/kdd/experiment/run.py --runner modal --modal-gpu T4 --modal-detach --mode real --backend offline --datasets locomo longmemeval memoryarena --max-conversations 999 --max-questions 999 --top-k 5 --eval-backend offline --visualize
Skipped Modal benchmark. Set RUN_MODAL_FULL_BENCHMARK=1 after Modal setup to execute.
Detached Modal runs print a call id; fetch with --modal-call-id <call-id>.


In [4]:
print("Optional external evaluator smoke commands:")
print("""
# Rhesis: set RHESIS_API_KEY outside the notebook first.
python experiment/run.py \\
  --mode real \\
  --eval-backend rhesis \\
  --eval-smoke-test \\
  --eval-limit 1

# Semantica: requires the semantica package to be installed.
python experiment/run.py \\
  --mode real \\
  --eval-backend semantica \\
  --eval-smoke-test \\
  --eval-limit 1
""")

Optional external evaluator smoke commands:

# Rhesis: set RHESIS_API_KEY outside the notebook first.
python experiment/run.py \
  --mode real \
  --eval-backend rhesis \
  --eval-smoke-test \
  --eval-limit 1

# Semantica: requires the semantica package to be installed.
python experiment/run.py \
  --mode real \
  --eval-backend semantica \
  --eval-smoke-test \
  --eval-limit 1



In [5]:
latest = sorted(RESULTS_DIR.glob("run_*_real_metrics.json"))[-1]
print("Reading", latest)
metrics = json.loads(latest.read_text())["real"]
for dataset, summary in metrics["datasets"].items():
    for strategy, row in summary["by_strategy"].items():
        print(dataset, strategy, row["questions"], row["retrieval_precision"], row["retrieval_recall"], row["evidence_hit_rate"])

Reading /Users/syaikhipin/Documents/Phd/Publish Paper/kdd/experiment/results/run_20260509_151530_real_metrics.json
LoCoMo no_memory 40 0.0 0.0 0.0
LoCoMo verbatim 40 0.07 0.3208 0.35
LoCoMo extracted_facts 40 0.075 0.3458 0.375
LoCoMo episodic 40 0.08 0.3708 0.4
LoCoMo hybrid 40 0.08 0.3708 0.4
LongMemEval no_memory 100 0.0 0.0 0.0
LongMemEval verbatim 100 1.0 0.9983 1.0
LongMemEval extracted_facts 100 1.0 0.9983 1.0
LongMemEval episodic 100 1.0 0.9983 1.0
LongMemEval hybrid 100 1.0 0.9983 1.0
MemoryArena no_memory 486 0.0 0.0 0.0
MemoryArena verbatim 486 0.2158 0.4095 0.4136
MemoryArena extracted_facts 486 0.5083 0.7181 0.7222
MemoryArena episodic 486 0.2117 0.3395 0.3436
MemoryArena hybrid 486 0.2158 0.4095 0.4136


Guidance: use this for Exercise 2. Participants compare strategies across datasets and explain why no single architecture dominates.